In [ ]:
# ==========================================
# YÊU CẦU 1: TF-IDF cho review_comment_message
# ==========================================

import pandas as pd
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
import joblib

# tải stopwords tiếng Bồ Đào Nha
nltk.download('stopwords', quiet=True)
stop_words_pt = stopwords.words('portuguese')

# đọc dữ liệu
df = pd.read_parquet('master_dataset.parquet')
print('Shape dữ liệu gốc:', df.shape)
print('Số cột:', len(df.columns))

# kiểm tra cột đầu vào bắt buộc
required_col = 'review_comment_message'
if required_col not in df.columns:
    raise ValueError("Không tìm thấy cột 'review_comment_message' trong dataset")

# kiểm thử dữ liệu trước khi đưa vào TF-IDF
missing_cnt = int(df[required_col].isna().sum())
empty_cnt = int((df[required_col].fillna('').astype(str).str.strip() == '').sum())
print(f"Số dòng null ở {required_col}: {missing_cnt}")
print(f"Số dòng rỗng ở {required_col}: {empty_cnt}")
print(f"Tỷ lệ rỗng/null: {(empty_cnt / len(df)):.2%}")

# xử lý null (giữ lại toàn bộ dòng)
df_reviews = df.copy()
df_reviews[required_col] = df_reviews[required_col].fillna('').astype(str)

# EDA nhanh cho text: phân phối độ dài review
df_reviews['review_len'] = df_reviews[required_col].str.len()
print('\nThống kê độ dài review (range/phân phối):')
print(df_reviews['review_len'].describe().to_string())

# xem mẫu dữ liệu trước khi vectorize
print('\n5 mẫu review trước xử lý TF-IDF:')
print(df_reviews[required_col].head(5).to_string(index=False))

# chia train/test 80/20 với seed cố định để tái lập kết quả
train_df, test_df = train_test_split(
    df_reviews,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

# Pipeline TF-IDF (đúng tiêu chí pipeline: fit/transform + save/load)
tfidf_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        stop_words=stop_words_pt,
        max_features=500,
        lowercase=True,
        min_df=5,
        token_pattern=r'(?u)\b[a-zA-ZÀ-ÿ]{2,}\b'
    ))
])

# fit trên train, transform cả train/test
X_train_tfidf = tfidf_pipeline.fit_transform(train_df[required_col])
X_test_tfidf = tfidf_pipeline.transform(test_df[required_col])

# thông tin kiểm tra tỉ lệ 80/20 + scale ma trận
n_total = len(df_reviews)
n_train = len(train_df)
n_test = len(test_df)
print(f"\nTotal: {n_total} | Train: {n_train} ({n_train / n_total:.2%}) | Test: {n_test} ({n_test / n_total:.2%})")
print('Kích thước ma trận TF-IDF train:', X_train_tfidf.shape)
print('Kích thước ma trận TF-IDF test:', X_test_tfidf.shape)
print('Top 20 từ khóa:', tfidf_pipeline.named_steps['tfidf'].get_feature_names_out()[:20])

# lưu và load lại pipeline TF-IDF
joblib.dump(tfidf_pipeline, 'tfidf_pipeline.joblib')
loaded_pipeline = joblib.load('tfidf_pipeline.joblib')
_ = loaded_pipeline.transform(test_df[required_col].head(3))
print('Đã lưu & load lại tfidf_pipeline.joblib thành công')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Kích thước ma trận TF-IDF: (113425, 500)
Top 20 từ khóa: ['abri' 'abrir' 'absurdo' 'acabamento' 'achei' 'acho' 'aconteceu' 'acordo'
 'acredito' 'adorei' 'agilidade' 'agora' 'agradeço' 'aguardando' 'aguardo'
 'ainda' 'algum' 'alguns' 'além' 'amei']
✅ Đã lưu tfidf_vectorizer.joblib


In [ ]:
# ==========================================
# YÊU CẦU 2: FP-Growth cho giỏ hàng
# ==========================================

import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

# đọc dữ liệu
df = pd.read_parquet('master_dataset.parquet')
print('Shape dữ liệu gốc:', df.shape)

# kiểm tra cột đầu vào bắt buộc
cat_candidates = ['product_category_name_english', 'product_category_name']
cat_col = next((c for c in cat_candidates if c in df.columns), None)
required_cols = ['order_id']

if 'order_id' not in df.columns:
    raise ValueError("Không tìm thấy cột 'order_id' trong dataset")
if cat_col is None:
    raise ValueError("Không tìm thấy cột category (product_category_name_english hoặc product_category_name)")

print('Cột category dùng cho FP-Growth:', cat_col)

# kiểm thử dữ liệu trước khi đưa vào FP-Growth
missing_order = int(df['order_id'].isna().sum())
missing_cat = int(df[cat_col].isna().sum())
print(f"Số dòng thiếu order_id: {missing_order}")
print(f"Số dòng thiếu {cat_col}: {missing_cat}")
print(f"Tỷ lệ thiếu {cat_col}: {(missing_cat / len(df)):.2%}")

dup_pairs = int(df[['order_id', cat_col]].dropna().duplicated().sum())
print('Số bản ghi trùng (order_id, category):', dup_pairs)

print('\n5 dòng dữ liệu gốc cho FP-Growth:')
print(df[['order_id', cat_col]].head(5).to_string(index=False))

# tiền xử lý: drop null và chuẩn hóa text category
df_items = df.dropna(subset=['order_id', cat_col]).copy()
df_items[cat_col] = df_items[cat_col].astype(str).str.strip()
df_items = df_items[df_items[cat_col] != '']

print('\nSố dòng sau làm sạch:', len(df_items))
print('Số order duy nhất:', df_items['order_id'].nunique())
print('Số category duy nhất:', df_items[cat_col].nunique())

# tạo transactions theo order
transactions = (
    df_items.groupby('order_id')[cat_col]
    .apply(lambda x: sorted(set(x)))
    .tolist()
)

# EDA nhanh: phân phối số item mỗi giao dịch
txn_len = pd.Series([len(t) for t in transactions], name='items_per_order')
print('\nThống kê số item mỗi order (trước lọc):')
print(txn_len.describe().to_string())

# lọc đơn có >=2 sản phẩm
transactions = [t for t in transactions if len(t) >= 2]
print('Đơn hàng có >=2 sản phẩm:', len(transactions))

if len(transactions) == 0:
    raise ValueError('Không có transaction nào đủ điều kiện (>=2 sản phẩm) để chạy FP-Growth')

# one-hot transaction matrix (không cần scale số)
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_trans = pd.DataFrame(te_ary, columns=te.columns_)
print('Kích thước ma trận transaction one-hot:', df_trans.shape)

# FP-Growth + Association Rules
frequent_itemsets = fpgrowth(df_trans, min_support=0.0001, use_colnames=True)
print('Frequent itemsets:', frequent_itemsets.shape)

if frequent_itemsets.empty:
    raise ValueError('Không tìm được frequent itemsets. Hãy tăng dữ liệu hoặc giảm min_support.')

rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=0.01)
print('Rules:', rules.shape)

if rules.empty:
    print('Không có luật nào thỏa confidence hiện tại. Hãy giảm min_threshold.')
    top_rules = pd.DataFrame(columns=['antecedents', 'consequents', 'support', 'confidence', 'lift'])
else:
    top_rules = rules.sort_values(by=['lift', 'confidence'], ascending=False).head(10).copy()

    # đổi sang string để CSV dễ đọc
    top_rules['antecedents'] = top_rules['antecedents'].apply(lambda x: ','.join(sorted(list(x))))
    top_rules['consequents'] = top_rules['consequents'].apply(lambda x: ','.join(sorted(list(x))))

print('\nTop 10 rules:')
print(top_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].to_string(index=False))

top_rules.to_csv('fp_growth_rules.csv', index=False)
print('Đã lưu fp_growth_rules.csv')

Đơn hàng có >=2 sản phẩm: 726


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Frequent itemsets: (317, 2)
Rules: (519, 14)

🏆 Top 10 rules:
                       antecedents                    consequents   support  \
491         fashio_female_clothing                  fashion_sport  0.001377   
490                  fashion_sport         fashio_female_clothing  0.001377   
177  fashion_bags_accessories,auto            musical_instruments  0.001377   
178            musical_instruments  fashion_bags_accessories,auto  0.001377   
515                          music                books_technical  0.001377   
516                books_technical                          music  0.001377   
176       musical_instruments,auto       fashion_bags_accessories  0.001377   
518      fashion_childrens_clothes       fashion_bags_accessories  0.002755   
517       fashion_bags_accessories      fashion_childrens_clothes  0.002755   
179       fashion_bags_accessories       musical_instruments,auto  0.001377   

     confidence        lift  
491    1.000000  363.000000  
490    0

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag